# Inspect Firestore collections

Print a few sample documents and check the size of a Firestore collection, working from
its local export (gzipped JSON lines, one document per line) rather than querying Firestore.

In [2]:
import gzip
import json
from collections import Counter
from pathlib import Path

EXPORT_PATH = Path("../profile_compensation_estimates.jsonl.gz")
N_SAMPLES = 3  # documents to print

with gzip.open(EXPORT_PATH, "rt", encoding="utf-8") as f:
    lines = [line for line in f if line.strip()]
records = [json.loads(line) for line in lines]

print(f"loaded {len(records):,} records from {EXPORT_PATH.name}")

loaded 12,450 records from profile_compensation_estimates.jsonl.gz


## Sample documents

In [3]:
def _truncate(value, max_len=200, max_items=2):
    """Shorten long strings and lists so a printed document stays readable."""
    if isinstance(value, str) and len(value) > max_len:
        return value[:max_len] + f"... (+{len(value) - max_len} chars)"
    if isinstance(value, list):
        head = [_truncate(v, max_len, max_items) for v in value[:max_items]]
        return head + [f"... (+{len(value) - max_items} more)"] if len(value) > max_items else head
    if isinstance(value, dict):
        return {k: _truncate(v, max_len, max_items) for k, v in value.items()}
    return value


for record in records[:N_SAMPLES]:
    print(f"--- {record.get('doc_id')} ---")
    print(json.dumps(_truncate(record), indent=2, ensure_ascii=False))

--- 000GQ42n5pvkJ0itmSjK ---
{
  "doc_id": "000GQ42n5pvkJ0itmSjK",
  "profile_id": "000GQ42n5pvkJ0itmSjK",
  "id": "000GQ42n5pvkJ0itmSjK",
  "company": "knight barry title group",
  "title": "title examiner",
  "totalMin": 40000,
  "totalMax": 80000,
  "explanation": "Candidate is a mid-career title examiner with 15 years' experience and long tenure at Knight Barry; market data for the same company/title supports the range. No additional pay typical for role or sen... (+27 chars)",
  "referenceData": [
    {
      "tc_min": 44011.375,
      "addl_pay_min": 0,
      "base_salary": 57967.46,
      "addl_pay_max": 0,
      "addl_pay": 0,
      "count": 14,
      "company": "knight barry title group",
      "tc_max": 76454.79,
      "search_relevance_score": 13,
      "dataSource": "minerva",
      "location": "united states",
      "base_salary_min": 44011.375,
      "total_comp": 57967.46,
      "title": "title examiner",
      "base_salary_max": 76454.79
    }
  ],
  "updatedAt": "2026-

## Collection size

In [4]:
doc_ids = [r.get("doc_id") for r in records]
# A resumed export can write the same page twice, so check for repeats.
print(f"records:         {len(records):,}")
print(f"unique doc_ids:  {len(set(doc_ids)):,}")

record_bytes = [len(line.encode("utf-8")) for line in lines]
total_bytes = sum(record_bytes)
print(f"compressed file: {EXPORT_PATH.stat().st_size / 1e6:,.1f} MB")
print(f"uncompressed:    {total_bytes / 1e6:,.1f} MB")
print(f"per record:      avg {total_bytes / len(lines):,.0f} B, min {min(record_bytes):,} B, max {max(record_bytes):,} B")

records:         12,450
unique doc_ids:  12,450
compressed file: 6.9 MB
uncompressed:    47.3 MB
per record:      avg 3,802 B, min 489 B, max 6,306 B


## Fields

Which fields appear, how often, and with what types.

In [5]:
field_types = {}
for record in records:
    for key, value in record.items():
        field_types.setdefault(key, Counter())[type(value).__name__] += 1

for key, types in field_types.items():
    print(f"{key:<15} {sum(types.values()):>7,}/{len(records):,}  {dict(types)}")

doc_id           12,450/12,450  {'str': 12450}
profile_id       12,450/12,450  {'str': 12450}
id               12,450/12,450  {'str': 12450}
company          12,450/12,450  {'str': 12450}
title            12,450/12,450  {'str': 12450}
totalMin         12,450/12,450  {'int': 12450}
totalMax         12,450/12,450  {'int': 12450}
explanation      12,450/12,450  {'str': 12450}
referenceData    12,450/12,450  {'list': 12450}
updatedAt        12,450/12,450  {'str': 12450}
extra                 2/12,450  {'dict': 2}


## Profile sample from OpenSearch

Analysis of `profile_sample_1000.csv`, the stratified sample written by
`common/sample_profiles_from_opensearch.py`. This section only needs the CSV, except the last
cell, which compares against `records` loaded at the top of the notebook.

In [6]:
import csv
import statistics

SAMPLE_PATH = Path("../profile_sample_1000.csv")

with open(SAMPLE_PATH, newline="", encoding="utf-8") as f:
    sample = list(csv.DictReader(f))

print(f"loaded {len(sample):,} profiles from {SAMPLE_PATH.name}")
print(f"unique profile_ids: {len({r['profile_id'] for r in sample}):,}")
print(f"unique companies:   {len({r['company'] for r in sample if r['company']}):,}")

loaded 1,000 profiles from profile_sample_1000.csv
unique profile_ids: 1,000
unique companies:   1,000


### Fill rates

How often each column has a value, and how many distinct values it takes.

In [7]:
for column in sample[0]:
    values = [r[column] for r in sample if r[column]]
    print(f"{column:<17} {len(values) / len(sample):>6.1%} filled  {len(set(values)):>5,} distinct")

profile_id        100.0% filled  1,000 distinct
company           100.0% filled  1,000 distinct
title             100.0% filled    785 distinct
role               84.1% filled     24 distinct
sub_role           67.0% filled     88 distinct
level              59.3% filled     10 distinct
levels             59.3% filled     21 distinct
years_experience   74.6% filled     50 distinct
yoe_bucket        100.0% filled      7 distinct
country            99.4% filled    189 distinct
region             66.1% filled    346 distinct
metro               4.5% filled     27 distinct
continent          99.3% filled      6 distinct
company_id         82.9% filled    829 distinct
company_size       81.9% filled      8 distinct
industry           79.3% filled    114 distinct
linkedin_url       94.8% filled    948 distinct


### Distribution per dimension

Empty values are shown as `(missing)`.

In [8]:
def show_distribution(rows, column, top=15, width=40):
    """Print the most common values of a column with their share and a text bar."""
    counts = Counter(r[column] or "(missing)" for r in rows)
    print(f"{column}: {len(counts):,} distinct values")
    largest = counts.most_common(1)[0][1]
    for value, count in counts.most_common(top):
        bar = "#" * max(1, round(width * count / largest))
        print(f"  {value[:30]:<30} {count:>5,} {count / len(rows):>6.1%}  {bar}")
    if len(counts) > top:
        rest = sum(count for _, count in counts.most_common()[top:])
        print(f"  ... {len(counts) - top:,} more values covering {rest / len(rows):.1%}")
    print()


for column in ["role", "level", "yoe_bucket", "company_size", "continent", "country", "industry"]:
    show_distribution(sample, column)

role: 25 distinct values
  (missing)                        159  15.9%  ########################################
  operations                        89   8.9%  ######################
  engineering                       61   6.1%  ###############
  professional_service              53   5.3%  #############
  education                         53   5.3%  #############
  hospitality                       47   4.7%  ############
  sales                             46   4.6%  ############
  fulfillment                       45   4.5%  ###########
  health                            45   4.5%  ###########
  finance                           40   4.0%  ##########
  creative                          38   3.8%  ##########
  analyst                           32   3.2%  ########
  support                           32   3.2%  ########
  human_resources                   31   3.1%  ########
  marketing                         31   3.1%  ########
  ... 10 more values covering 19.8%

level: 11 distinc

### Years of experience

In [9]:
yoe = [int(r["years_experience"]) for r in sample if r["years_experience"]]
quartiles = statistics.quantiles(yoe, n=4)

print(f"profiles with years_experience: {len(yoe):,} of {len(sample):,}")
print(f"min {min(yoe)}, p25 {quartiles[0]:.0f}, median {quartiles[1]:.0f}, p75 {quartiles[2]:.0f}, max {max(yoe)}, mean {statistics.mean(yoe):.1f}")

profiles with years_experience: 746 of 1,000
min 0, p25 6, median 13, p75 19, max 59, mean 13.8


### Role by level

Counts for each role and level pair, to see where the sample is thin. Rows are the most common
roles; `-` marks a pair with no profiles.

In [10]:
LEVELS = ["entry", "training", "senior", "manager", "director", "vp", "partner", "owner", "cxo", "unpaid", ""]
TOP_ROLES = 12

pairs = Counter((r["role"], r["level"]) for r in sample)
roles = [role for role, _ in Counter(r["role"] for r in sample).most_common(TOP_ROLES)]

print(f"{'role':<22}" + "".join(f"{(level or '(none)')[:8]:>9}" for level in LEVELS) + f"{'total':>8}")
for role in roles:
    row = [pairs[(role, level)] for level in LEVELS]
    print(f"{(role or '(missing)')[:22]:<22}" + "".join(f"{n or '-':>9}" for n in row) + f"{sum(row):>8}")

known = [(role, level) for (role, level) in pairs if role and level]
print(f"\n{len(known)} distinct role and level pairs have at least one profile")

role                      entry training   senior  manager director       vp  partner    owner      cxo   unpaid   (none)   total
(missing)                     6        3        6        5       17       11        8        1        5        8       89     159
operations                    1        -        -       14        8        1        3       26       27        -        9      89
engineering                   2        3       14        7        1        3        -        4        2        -       25      61
professional_service          4        3        6       12        6        4        9        2        2        -        5      53
education                     -        3        5        -        -        2        -        3        -        1       39      53
hospitality                   -        -        2       16        5        -        -        9        2        -       13      47
sales                         -        -        4       15        5        3        -     

### Profiles that may not have a real current job

Titles such as "retired" or "student" pass the sampler's "has a title and company" filter but are
poor test cases for compensation. The keyword list is a starting point, not a complete one.

In [11]:
import re

NOT_EMPLOYED_WORDS = ["retired", "unemployed", "student", "intern", "volunteer", "seeking", "looking for", "freelance", "self employed", "self-employed", "stay at home", "homemaker"]


def not_employed_reason(row):
    if row["sub_role"] == "unemployed":
        return "sub_role=unemployed"
    if row["level"] in ("training", "unpaid"):
        return f"level={row['level']}"
    for word in NOT_EMPLOYED_WORDS:
        # Whole words only, so "intern" does not match "international".
        if re.search(rf"\b{re.escape(word)}\b", row["title"].lower()):
            return f"title contains '{word}'"
    return None


flagged = [(reason, r) for r in sample if (reason := not_employed_reason(r))]
print(f"{len(flagged):,} of {len(sample):,} profiles flagged ({len(flagged) / len(sample):.1%})\n")
for reason, count in Counter(reason for reason, _ in flagged).most_common():
    print(f"  {reason:<32} {count:>4}")
print()
for reason, r in flagged[:10]:
    print(f"  {r['title'][:40]:<40} | {r['company'][:30]:<30} | {reason}")

68 of 1,000 profiles flagged (6.8%)

  level=training                     29
  sub_role=unemployed                20
  level=unpaid                       11
  title contains 'student'            7
  title contains 'freelance'          1

  student intern                           | london business school         | level=training
  volunteer                                | cxk                            | level=unpaid
  unemployed                               | envoy group                    | sub_role=unemployed
  legal intern                             | yavuz uyanık attorney partners | level=training
  student consultant                       | randstad                       | title contains 'student'
  unemployed                               | no company                     | sub_role=unemployed
  marketing intern                         | hireclix                       | level=training
  volunteer in offices                     | underwood memorial hospital    | level=unpaid
  

### Overlap with the compensation estimates

How many sampled profiles already have a cached estimate in the Firestore export loaded above.

In [12]:
estimate_ids = {r.get("profile_id") for r in records} | {r.get("doc_id") for r in records}
sample_ids = {r["profile_id"] for r in sample}
overlap = sample_ids & estimate_ids

print(f"sampled profiles with an existing estimate: {len(overlap):,} of {len(sample_ids):,}")
print(f"example sample id:   {next(iter(sample_ids))}")
print(f"example estimate id: {records[0].get('profile_id')}")

sampled profiles with an existing estimate: 0 of 1,000
example sample id:   Z4kUoyiFlQYpcHdHZ0F8ow_0000
example estimate id: 000GQ42n5pvkJ0itmSjK
